# Cluster Then Classify — Phases 0–2
### AG News (test split, 7,600 rows)

**Ground rule for this notebook: the true labels are quarantined in Phase 0 and
never touched again until Phase 5.** Every decision here — k, algorithm,
cluster names — must be justifiable without them. If you peek, the final
accuracy number means nothing.

In [ ]:
import os, re, html, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)
RANDOM_STATE = 42

---
## Phase 0 — Load, clean, quarantine the labels

In [ ]:
raw = pd.read_csv('test.csv')
print(raw.shape)
raw.head()

In [ ]:
# AG News class index -> name. Recorded here for Phase 5 only.
CLASS_NAMES = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
print(raw['Class Index'].value_counts().sort_index().rename(CLASS_NAMES))

In [ ]:
# Source markers, in one place. Left in the text these dominate the embedding
# and clusters form around the news agency instead of the topic.
SOURCES = (
    r"AP|Reuters|AFP|Forbes\.com|USATODAY\.com|SPACE\.com|NewsFactor|Ziff Davis|"
    r"PC World|InfoWorld|TechWeb|CNET|washingtonpost\.com|The Motley Fool|"
    r"Investor's Business Daily|Canadian Press|MacCentral|CBS MarketWatch\.com|AP Online"
)


def clean_field(s):
    """Applied to Title and Description SEPARATELY, then joined.

    Order matters. Cleaning the joined string does not work: the source-prefix
    patterns anchor to the start of a field, and once joined the Description no
    longer starts the string.
    """
    # 1. Literal backslash escapes that survived the original export
    s = s.replace("\\\\", " ").replace("\\", " ")

    # 2. Entities here are often missing their leading "&" ("#39;", "quot;").
    #    Repair first, or html.unescape silently leaves them in place.
    s = re.sub(r"(?<!&)#(\d+);", r"&#\1;", s)
    s = re.sub(r"(?<!&)\b(quot|amp|lt|gt|nbsp|apos);", r"&\1;", s)
    s = html.unescape(html.unescape(s))          # double-encoded in places

    # 3. Unescaping exposes real markup. Business stories carry Reuters quote
    #    links: <A HREF="http://investor.reuters.com/FullQuote.aspx?ticker=...">
    #    Left in, the Business cluster forms around "fullquote", "aspx" and
    #    "reuters" instead of around business content.
    s = re.sub(r"<[^>]{0,400}>", " ", s)
    s = re.sub(r"\b[A-Z]{1,5}\.[A-Z]{1,3}\b", " ", s)      # ticker symbols, SPLS.O

    # 4. Source markers, in all three positions they occur
    s = re.sub(r"^\s*(" + SOURCES + r")\s*[-\u2013:]\s*", "", s, flags=re.I)
    s = re.sub(r"^\s*[A-Z][A-Za-z .]{2,25}\s*\((" + SOURCES + r")\)\s*[-\u2013]\s*", "", s)
    s = re.sub(r"\s*\((" + SOURCES + r")\)", " ", s, flags=re.I)

    s = re.sub(r"\s+([.,;:!?])", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()


title = raw["Title"].map(clean_field)
desc = raw["Description"].map(clean_field)

df = raw.copy()
df["text"] = (title + ". " + desc).str.replace(r"\s+", " ", regex=True).str.strip()

before = len(df)
df = df[df["text"].str.split().str.len() >= 5]          # gutted by cleaning
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print("dropped", before - len(df), "rows (duplicate or too short);", len(df), "remain")

df["text"].head(3).tolist()

In [ ]:
# Verify cleaning did what it claims. Each of these caught a real bug during
# development -- the html-tag and source-marker rows especially.
checks = {
    "entities":        r"#\d+;|&[a-z]+;",
    "backslashes":     r"\\",
    "html tags":       r"<[^>]+>",
    "(AP)/(Reuters)":  r"\((?:AP|Reuters|AFP)\)",
    "reuters boiler":  r"(?i)fullquote|aspx",
}
for name, pat in checks.items():
    print(f"residual {name:16}: {df['text'].str.contains(pat, regex=True).sum()}")

lens = df["text"].str.split().str.len()
print(f"\nword count: median {lens.median():.0f}, p95 {lens.quantile(0.95):.0f}, max {lens.max()}")
print("-> comfortably under MiniLM's 256-token cap, so nothing gets truncated")

In [ ]:
# ============================================================
# QUARANTINE. Split the labels off to disk and drop them from
# the working frame. Do not load this file again until Phase 5.
# ============================================================
df[['Class Index']].to_csv('_true_labels_DO_NOT_OPEN_UNTIL_PHASE_5.csv', index=False)

work = df[['text']].copy()          # the only frame used from here on
del df, raw                          # remove the labels from the session entirely

print(work.shape)
work.head()

---
## Phase 1 — Embeddings

Computed once, cached to disk. You will re-run the clustering below dozens of
times; recomputing embeddings each pass wastes an hour a day for nothing.

In [ ]:
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer

EMB_PATH = 'embeddings_minilm.npy'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

if os.path.exists(EMB_PATH):
    emb = np.load(EMB_PATH)
    print(f'loaded cached embeddings {emb.shape}')
else:
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(
        work['text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,   # makes euclidean distance == cosine distance
    )
    np.save(EMB_PATH, emb)
    print(f'computed and cached {emb.shape}')

---
## Phase 2 — Clustering

**Two days, hard stop.** The gate at the end of this phase is not optional.

In [ ]:
# --- k sweep -------------------------------------------------------------
# Silhouette is a *geometric* measure on 384-dim data. Expect small values and
# expect it to drift upward with k. It is a weak signal here -- recorded for the
# writeup, not used to choose k. The manual read three cells down decides.
ks = [2, 3, 4, 5, 6, 7, 8, 10, 12, 15, 20]
rows = []
labels_by_k = {}

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lab = km.fit_predict(emb)
    labels_by_k[k] = lab
    rows.append({'k': k, 'inertia': km.inertia_,
                 'silhouette': silhouette_score(emb, lab, sample_size=5000,
                                                random_state=RANDOM_STATE)})

sweep = pd.DataFrame(rows)
sweep

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(sweep['k'], sweep['inertia'], 'o-'); ax[0].set(xlabel='k', ylabel='inertia', title='Elbow')
ax[1].plot(sweep['k'], sweep['silhouette'], 'o-'); ax[1].set(xlabel='k', ylabel='silhouette', title='Silhouette')
plt.tight_layout(); plt.show()

print('If silhouette climbs steadily with k, it is measuring granularity, not')
print('correctness. Do not let it pick k for you.')

In [ ]:
# --- UMAP projection, for looking at ------------------------------------
# pip install umap-learn
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                    random_state=RANDOM_STATE)
proj = reducer.fit_transform(emb)
np.save('umap_2d.npy', proj)
print(proj.shape)

In [ ]:
K = 4   # starting hypothesis; revise after reading the samples below
lab = labels_by_k[K]

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
ax[0].scatter(proj[:, 0], proj[:, 1], s=2, alpha=0.3, c='0.6')
ax[0].set_title('Embedding space (unlabeled)')
sc = ax[1].scatter(proj[:, 0], proj[:, 1], s=2, alpha=0.5, c=lab, cmap='tab10')
ax[1].set_title(f'KMeans, k={K}')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

print(pd.Series(lab).value_counts().sort_index().rename('size'))

### The manual read

The step people skip. A cluster can score well geometrically and still mean
nothing. Read the samples and write down, in your own words, what each cluster
is about — **before** the LLM sees anything in Phase 3. Your descriptions are
what you check the LLM's labels against.

In [ ]:
def cluster_samples(labels, k, n=10, mode='central'):
    """mode='central' -> closest to centroid (the cluster's core)
       mode='random'  -> uniform sample (shows you the messy edges)"""
    out = {}
    for c in range(k):
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            out[c] = []
            continue
        if mode == 'central':
            centroid = emb[idx].mean(axis=0, keepdims=True)
            order = idx[np.argsort(-cosine_similarity(emb[idx], centroid).ravel())]
        else:
            order = np.random.RandomState(RANDOM_STATE).permutation(idx)
        out[c] = work['text'].iloc[order[:n]].tolist()
    return out


samples = cluster_samples(lab, K, n=10, mode='central')
for c, texts in samples.items():
    print('=' * 90)
    print(f'CLUSTER {c}   (n={(lab == c).sum()})')
    print('=' * 90)
    for t in texts:
        print('  -', t[:160])
    print()

In [ ]:
# Also read a random sample. Central docs always look coherent; the edges are
# where you find out whether the cluster is actually one thing.
edges = cluster_samples(lab, K, n=6, mode='random')
for c, texts in edges.items():
    print(f'--- CLUSTER {c} (random) ---')
    for t in texts:
        print('  -', t[:140])
    print()

In [ ]:
# --- distinctive terms per cluster, as a cross-check on your reading -----
tf = TfidfVectorizer(max_features=20000, min_df=5, stop_words='english')
M = tf.fit_transform(work['text'])
vocab = np.array(tf.get_feature_names_out())

for c in range(K):
    mask = (lab == c)
    inside = np.asarray(M[mask].mean(axis=0)).ravel()
    outside = np.asarray(M[~mask].mean(axis=0)).ravel()
    top = vocab[np.argsort(-(inside - outside))[:12]]
    print(f'cluster {c}: {", ".join(top)}')

In [ ]:
# WRITE YOUR OWN DESCRIPTIONS HERE, from the samples above. One line each.
# These get compared against the LLM's labels in Phase 3 and against the true
# classes in Phase 5.
my_descriptions = {
    0: '',
    1: '',
    2: '',
    3: '',
}

assert all(v.strip() for v in my_descriptions.values()), \
    'Fill these in by reading the samples. This is the Phase 2 deliverable.'

json.dump(my_descriptions, open('phase2_human_descriptions.json', 'w'), indent=2)
my_descriptions

### Alternative: HDBSCAN

Finds its own cluster count and marks ambiguous points as noise (label `-1`)
instead of forcing every document into a bucket. Often a more honest picture of
whether the structure is really there.

In [ ]:
# pip install hdbscan
import hdbscan

# Cluster in a reduced space -- HDBSCAN's density estimates degrade badly in 384 dims.
dens = umap.UMAP(n_components=15, n_neighbors=15, min_dist=0.0,
                 metric='cosine', random_state=RANDOM_STATE).fit_transform(emb)

hdb = hdbscan.HDBSCAN(min_cluster_size=80, min_samples=10,
                      metric='euclidean', cluster_selection_method='eom')
hlab = hdb.fit_predict(dens)

n_clusters = len(set(hlab)) - (1 if -1 in hlab else 0)
print(f'clusters found: {n_clusters}')
print(f'noise points  : {(hlab == -1).sum()} ({(hlab == -1).mean():.1%})')
print(pd.Series(hlab).value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
noise = hlab == -1
ax.scatter(proj[noise, 0], proj[noise, 1], s=2, c='0.85', label='noise')
ax.scatter(proj[~noise, 0], proj[~noise, 1], s=2, c=hlab[~noise], cmap='tab20')
ax.set_xticks([]); ax.set_yticks([]); ax.set_title('HDBSCAN'); ax.legend(loc='upper right')
plt.show()

---
## Decision gate — end of Day 4

Answer these before writing a line of Phase 3.

1. **Can you name every cluster in one sentence from the samples alone?**
   If any cluster reads as "assorted news," the pipeline downstream inherits
   that and you will not find out until Day 9.

2. **Do the cluster sizes look plausible?** One cluster holding 70% of the
   corpus means k is too low or the embedding is not separating.

3. **Does KMeans agree with HDBSCAN about the broad structure?** They need not
   match exactly, but they should not be telling opposite stories.

**A note specific to this dataset.** A TF-IDF dry run on these 7,600 rows merged
Business and Sci/Tech into a single cluster (≈1,490 + ≈1,350 documents together)
while Sports and World separated cleanly. Sentence embeddings should improve on
that, but expect the pair to stay entangled — a story about a chip maker's
quarterly earnings is honestly both. **This is not a Phase 2 failure and it is
not a reason to stop.** It is the finding you write up in Phase 5, and it is the
reason `k=5` or `k=6` is worth testing: a split that separates *corporate*
tech from *research* tech may map onto the true classes better than forcing
k=4. Let the manual read decide, not the assumption that k should equal the
number of true classes.

Stop only if clusters are unreadable — not if they are readable but do not match
the four categories you happen to know are underneath.

**Deliverables before moving on:** cached embeddings, a chosen algorithm and k,
`phase2_human_descriptions.json` filled in by you, and the UMAP plot saved.

In [ ]:
# Freeze the Phase 2 choice for Phase 3 to pick up.
FINAL_K = K
final_labels = labels_by_k[FINAL_K]

np.save('cluster_labels.npy', final_labels)
work.assign(cluster=final_labels).to_csv('phase2_clustered.csv', index=False)
print(f'saved k={FINAL_K}, {len(work)} rows')